# 🎓 MentorX — DeepSeekCoder-6.7B-Instruct SFT (QLoRA)

**Model:** deepseek-ai/deepseek-coder-6.7b-instruct  
**Yöntem:** QLoRA (4-bit) via Unsloth  
**Veri:** ~765 Türkçe Sokratik diyalog (Python CS1)  
**Şartlar:** StarCoder2 ile aynı — LORA_R=8, MAX_SEQ_LENGTH=1024, train_on_responses_only YOK

> ⚠️ **Colab Pro / A100 (40GB) önerilir.**

## 📦 1. Kurulum

In [1]:
%%capture
!pip install -U unsloth wandb scikit-learn

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 🔧 2. Konfigürasyon

In [3]:
# Model
MODEL_NAME       = "deepseek-ai/deepseek-coder-6.7b-instruct"
MAX_SEQ_LENGTH   = 1024
DTYPE            = None
LOAD_IN_4BIT     = True

# LoRA — StarCoder2 ile aynı
LORA_R           = 8
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.0
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

# Eğitim
OUTPUT_DIR       = "./mentorx-deepseek-coder-6.7b-lora"
NUM_EPOCHS       = 5
BATCH_SIZE       = 4
GRAD_ACCUM       = 4
LEARNING_RATE    = 1e-4
WARMUP_STEPS     = 30
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0
LOGGING_STEPS    = 10
SAVE_STEPS       = 20
EVAL_STEPS       = 20
SEED             = 42

# Veri
INPUT_FILE       = "train_py_cs1.jsonl"
TRAIN_FILE       = "train_py_cs1_train.jsonl"
VAL_FILE         = "train_py_cs1_val.jsonl"
VAL_RATIO        = 0.15

# HuggingFace Hub
HF_PUSH          = True
HF_REPO          = "mtepe01/mentorx-deepseek-coder-6.7b"

# WandB
USE_WANDB        = False
WANDB_PROJECT    = "mentorx-sft-python"

print("✅ Konfigürasyon yüklendi")
print(f"   Model        : {MODEL_NAME}")
print(f"   LORA_R       : {LORA_R} | LORA_ALPHA: {LORA_ALPHA}")
print(f"   Effective BS : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   train_on_responses_only: KAPALI (StarCoder2 ile aynı)")

✅ Konfigürasyon yüklendi
   Model        : deepseek-ai/deepseek-coder-6.7b-instruct
   LORA_R       : 8 | LORA_ALPHA: 16
   Effective BS : 16
   train_on_responses_only: KAPALI (StarCoder2 ile aynı)


## 📂 3. Veri Yükleme & Stratified Train/Val Split

In [4]:
from google.colab import files

print("train_py_cs1.jsonl dosyasını seç:")
uploaded = files.upload()

for fname in uploaded:
    print(f"✅ Yüklendi: {fname} ({len(uploaded[fname])/1024:.1f} KB)")

train_py_cs1.jsonl dosyasını seç:


Saving train_py_cs1.jsonl to train_py_cs1.jsonl
✅ Yüklendi: train_py_cs1.jsonl (1976.9 KB)


In [5]:
import json
from collections import Counter
from sklearn.model_selection import train_test_split

with open(INPUT_FILE, encoding="utf-8") as f:
    all_records = [json.loads(line) for line in f if line.strip()]

print(f"Toplam diyalog: {len(all_records)}")

profiles = [r.get("profile", "UNKNOWN") for r in all_records]
print(f"Profil dağılımı: {Counter(profiles)}")

train_records, val_records = train_test_split(
    all_records,
    test_size    = VAL_RATIO,
    random_state = SEED,
    stratify     = profiles,
)

print(f"\nTrain: {len(train_records)} | Val: {len(val_records)}")

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(VAL_FILE, "w", encoding="utf-8") as f:
    for r in val_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"✅ {TRAIN_FILE} ve {VAL_FILE} oluşturuldu")

Toplam diyalog: 765
Profil dağılımı: Counter({'UNKNOWN': 765})

Train: 650 | Val: 115
✅ train_py_cs1_train.jsonl ve train_py_cs1_val.jsonl oluşturuldu


In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

print(f"Train örnekleri : {len(dataset['train'])}")
print(f"Val örnekleri   : {len(dataset['validation'])}")

def validate_split(ds, name):
    for i, row in enumerate(ds):
        convs = row.get("conversations")
        assert isinstance(convs, list) and len(convs) >= 2, f"{name}[{i}] invalid"
        assert convs[-1].get("from") == "gpt", f"{name}[{i}] last must be gpt"
        for j, m in enumerate(convs):
            assert m.get("from") in {"system", "human", "gpt"}, f"{name}[{i}] bad role @{j}"
            v = m.get("value")
            assert isinstance(v, str) and v.strip(), f"{name}[{i}] empty @{j}"

validate_split(dataset["train"],      "train")
validate_split(dataset["validation"], "validation")
print("✅ Dataset şeması doğrulandı")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train örnekleri : 650
Val örnekleri   : 115
✅ Dataset şeması doğrulandı


## 🤖 4. Model Yükleme

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

print(f"✅ Model yüklendi: {MODEL_NAME}")
print(f"   Model vocab size : {model.config.vocab_size}")
print(f"   Tokenizer vocab  : {len(tokenizer)}")
print(f"   Parametre sayısı : {model.num_parameters()/1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

✅ Model yüklendi: deepseek-ai/deepseek-coder-6.7b-instruct
   Model vocab size : 32256
   Tokenizer vocab  : 32022
   Parametre sayısı : 6.74B


## 🔗 5. LoRA Adapter Ekleme

In [8]:
# Katman adlarını doğrula
linear_names = set()
for name, module in model.named_modules():
    if hasattr(module, 'weight') and 'Linear' in type(module).__name__:
        short = name.split('.')[-1]
        linear_names.add(short)

missing = [m for m in TARGET_MODULES if m not in linear_names]
if missing:
    raise ValueError(f"TARGET_MODULES modelde yok: {missing}")
print(f"✅ Tüm TARGET_MODULES mevcut: {TARGET_MODULES}")

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
)

print("✅ LoRA adapter eklendi")
model.print_trainable_parameters()

✅ Tüm TARGET_MODULES mevcut: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapter eklendi
trainable params: 19,988,480 || all params: 6,760,501,248 || trainable%: 0.2957


## 🔄 6. Veri Formatlama

In [9]:
print(f"Tokenizer vocab  : {len(tokenizer)}")
print(f"Model vocab size : {model.config.vocab_size}")

# DeepSeekCoder: model vocab > tokenizer vocab durumunda
# resize yerine config güncelle — embedding bozulmaz
if len(tokenizer) != model.config.vocab_size:
    print(f"⚠️  Vocab uyumsuzluğu: tokenizer={len(tokenizer)}, model={model.config.vocab_size}")
    if model.config.vocab_size > len(tokenizer):
        # Model daha büyük — config'i tokenizer'a eşitle
        model.config.vocab_size = len(tokenizer)
        print(f"✅ model.config.vocab_size → {len(tokenizer)}")
    else:
        # Tokenizer daha büyük — embedding resize et
        model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
        print(f"✅ Embedding resize → {len(tokenizer)}")
else:
    print("✅ Vocab boyutu eşleşiyor")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        messages = []
        for m in convo:
            role_map = {"system": "system", "human": "user", "gpt": "assistant"}
            messages.append({
                "role": role_map[m["from"]],
                "content": m["value"]
            })
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

print("✅ Veri formatlandı")
print("\n--- Örnek (ilk 300 karakter) ---")
print(dataset["train"][0]["text"][:300])

Tokenizer vocab  : 32022
Model vocab size : 32256
⚠️  Vocab uyumsuzluğu: tokenizer=32022, model=32256
✅ model.config.vocab_size → 32022


Map:   0%|          | 0/650 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

✅ Veri formatlandı

--- Örnek (ilk 300 karakter) ---
<｜begin▁of▁sentence｜>Sen bir Python programlama tutor'ısın. Öğrenciye Sokratik yöntemle rehberlik edersin. Tüm diyalog Türkçe olmalı — TypeError, for, range gibi teknik terimler ve kod keyword'leri hariç.

Kesinlikle yapmaman gerekenler:
- Kodu doğrudan yazma veya tamamlama
- Hatayı doğrudan gösterm


## 🏋️ 7. Eğitim

In [10]:
import os

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name="deepseek-coder-6.7b-python-cs1")
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("WandB devre dışı")

WandB devre dışı


In [11]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset["train"],
    eval_dataset       = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = SFTConfig(
        per_device_train_batch_size  = BATCH_SIZE,
        per_device_eval_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_steps                 = WARMUP_STEPS,
        num_train_epochs             = NUM_EPOCHS,
        learning_rate                = LEARNING_RATE,
        max_grad_norm                = MAX_GRAD_NORM,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = LOGGING_STEPS,
        save_steps                   = SAVE_STEPS,
        eval_strategy                = "steps",
        eval_steps                   = EVAL_STEPS,
        load_best_model_at_end       = True,
        metric_for_best_model        = "eval_loss",
        optim                        = "adamw_8bit",
        weight_decay                 = WEIGHT_DECAY,
        lr_scheduler_type            = LR_SCHEDULER,
        seed                         = SEED,
        output_dir                   = OUTPUT_DIR,
        report_to                    = "wandb" if USE_WANDB else "none",
        dataset_text_field           = "text",
        max_seq_length               = MAX_SEQ_LENGTH,
    ),
)

print("✅ Trainer hazır, eğitim başlıyor...")
trainer_stats = trainer.train()

print(f"\n✅ Eğitim tamamlandı!")
print(f"   Toplam adım    : {trainer_stats.global_step}")
print(f"   Eğitim süresi  : {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"   Son train loss : {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/650 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/115 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ Trainer hazır, eğitim başlıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 650 | Num Epochs = 5 | Total steps = 205
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 19,988,480 of 6,760,501,248 (0.30% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
20,13.336249,10.096771
40,5.492582,4.602284
60,3.319246,2.808340
80,2.151192,2.025318
100,1.810677,1.760393
120,1.650383,1.605018
140,1.517639,1.518230
160,1.472943,1.470962
180,1.442441,1.451182
200,1.425712,1.447135


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-deepseek-coder-6.7b-lora/checkpoint-20/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-deepseek-coder-6.7b-lora/checkpoint-40/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-deepseek-coder-6.7b-lora/checkpoint-60/tokenizer_config.json.
/usr/local/lib/pyth


✅ Eğitim tamamlandı!
   Toplam adım    : 205
   Eğitim süresi  : 1344.1s
   Son train loss : 3.6059


## 💾 8. Model Kaydetme & HF Push

In [12]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA ağırlıkları kaydedildi: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ./mentorx-deepseek-coder-6.7b-lora/tokenizer_config.json.


✅ LoRA ağırlıkları kaydedildi: ./mentorx-deepseek-coder-6.7b-lora


In [14]:
if HF_PUSH:
    from huggingface_hub import login
    login()
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f"✅ HF Hub'a yüklendi: https://huggingface.co/{HF_REPO}")
else:
    print("HF_PUSH=False — Hub'a yükleme atlandı")

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 49.2kB /  608MB            

Saved model to https://huggingface.co/mtepe01/mentorx-deepseek-coder-6.7b


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmps3dhf7qg/tokenizer_config.json.
No files have been modified since last commit. Skipping to prevent empty commit.


✅ HF Hub'a yüklendi: https://huggingface.co/mtepe01/mentorx-deepseek-coder-6.7b


In [15]:
model.save_pretrained_merged(
    OUTPUT_DIR + "-merged",
    tokenizer,
    save_method = "merged_16bit",
)

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(HF_REPO + "-merged", exist_ok=True)
model.push_to_hub_merged(
    HF_REPO + "-merged",
    tokenizer,
    save_method = "merged_16bit",
)
print(f"✅ Merged model push edildi: https://huggingface.co/{HF_REPO}-merged")

Unsloth: Restored added_tokens_decoder metadata in ./mentorx-deepseek-coder-6.7b-lora-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `./mentorx-deepseek-coder-6.7b-lora-merged`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `./mentorx-deepseek-coder-6.7b-lora-merged`:  50%|█████     | 1/2 [00:19<00:19, 19.08s/it]
Unsloth: Copying 2 files from cache to `./mentorx-deepseek-coder-6.7b-lora-merged`: 100%|██████████| 2/2 [00:28<00:00, 14.22s/it]


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   5%|4         |  160MB / 3.50GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:22<00:00, 101.43s/it]


Unsloth: Merge process complete. Saved to `/content/mtepe01/mentorx-deepseek-coder-6.7b-merged`
✅ Merged model push edildi: https://huggingface.co/mtepe01/mentorx-deepseek-coder-6.7b-merged


## 🧪 9. Davranış Testleri (Inference)

In [16]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Sen MentorX adlı bir Türkçe Python programlama tutorusun. "
    "Öğrenciye asla doğrudan cevap vermezsin, her mesajında yönlendirici sorular sorarsın, "
    "max 3 cümleyle konuşursun."
)

def ask(user_message, history=None, max_new_tokens=200):
    if history is None:
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
    history.append({"role": "user", "content": user_message})
    inputs = tokenizer.apply_chat_template(
        history,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": response})
    return response, history

print("✅ Inference fonksiyonu hazır")

✅ Inference fonksiyonu hazır


In [17]:
print("=" * 60)
print("TEST 1: Sokratik davranış")
print("=" * 60)
resp, _ = ask("Liste ve tuple arasındaki fark nedir?")
print(f"👤 Öğrenci: Liste ve tuple arasındaki fark nedir?")
print(f"🎓 MentorX: {resp}\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 1: Sokratik davranış


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


👤 Öğrenci: Liste ve tuple arasındaki fark nedir?
🎓 MentorX: Kodundaneyazmaveyadaikideğerleridüzeltekbudeğitirmemgerekiyorum.Pekibirlisteyapabilirim?



In [18]:
print("=" * 60)
print("TEST 2: Direkt cevap isteği")
print("=" * 60)
resp, _ = ask("Bana cevabı direkt söyle, vakit kaybetmek istemiyorum.")
print(f"👤 Öğrenci: Bana cevabı direkt söyle, vakit kaybetmek istemiyorum.")
print(f"🎓 MentorX: {resp}\n")

Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 2: Direkt cevap isteği
👤 Öğrenci: Bana cevabı direkt söyle, vakit kaybetmek istemiyorum.
🎓 MentorX: Doğrutespitettin.Pekikodunluaralıyorsanaziklipucumandavramısadeceğinihatırlatanlatırsanönceceğim.İpuanbirsözlüktekiğineolabilirmisin?



In [19]:
print("=" * 60)
print("TEST 3: Çok turlu diyalog")
print("=" * 60)

history = None
turns = [
    "for döngüsü nedir?",
    "Belirli sayıda tekrar eden bir yapı mı?",
    "Peki while döngüsünden farkı ne?",
]

for q in turns:
    resp, history = ask(q, history)
    print(f"👤 Öğrenci: {q}")
    print(f"🎓 MentorX: {resp}")
    print("-" * 40)

Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 3: Çok turlu diyalog


Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


👤 Öğrenci: for döngüsü nedir?
🎓 MentorX: Güzelbirtipucamangerekenlerinikullanmalıyorum.Pekibumdafor'dabloolur.İpucarametresiyenitelemanlarvekodundahataalıyorsun?
----------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=16384) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


👤 Öğrenci: Belirli sayıda tekrar eden bir yapı mı?
🎓 MentorX: Evet,Python'dabloğrudangöndericiçalımağınıdüünelim.Aynenfordöngüsütekrarlayabilirmisin?
----------------------------------------
👤 Öğrenci: Peki while döngüsünden farkı ne?
🎓 MentorX: Doğrutespitettin.Pekidüzelttensonra,`if`gibibireyyazdırmakgerekiyormusun?
----------------------------------------
